In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [25]:
X_train = pd.read_csv("../data/X_rf_train.csv")
X_test = pd.read_csv("../data/X_rf_test.csv")
y_train = pd.read_csv("../data/y_train.csv")
y_test = pd.read_csv("../data/y_test.csv")

In [26]:
y_train = y_train.squeeze()
y_test = y_test.squeeze()

In [27]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

In [28]:
X_train_scaled=scaler.fit_transform(X_train)
X_test_scaled=scaler.transform(X_test)

In [19]:
from sklearn.svm import SVR
svr_linear = SVR(kernel='linear', C=100, epsilon=0.1)
svr_linear.fit(X_train_scaled, y_train)

y_pred_linear = svr_linear.predict(X_test_scaled)

In [20]:
from sklearn.metrics import(r2_score,mean_absolute_error,mean_squared_error)
r2=r2_score(y_test,y_pred_linear)
mae = mean_absolute_error(y_test,y_pred_linear)
rmse = np.sqrt(mean_squared_error(y_test,y_pred_linear))

In [21]:
y_train_pred = svr_linear.predict(X_train_scaled)
train_r2 = r2_score(y_train,y_train_pred)
print("Train R2:", round(train_r2,4))

Train R2: 0.3888


In [22]:
print("Test Metrics")
print("r2: ",r2)
print("mae: ",mae)
print("rmse: ",rmse)


Test Metrics
r2:  0.45770625019155586
mae:  0.17487949340947986
rmse:  0.4226127233134234


In [23]:
from sklearn.svm import SVR

In [41]:
svr_rbf = SVR(kernel='rbf', C=100, gamma=0.1, epsilon=0.01)
svr_rbf.fit(X_train_scaled, y_train)

y_pred_rbf = svr_rbf.predict(X_test_scaled)

In [42]:
from sklearn.metrics import(r2_score,mean_absolute_error,mean_squared_error)
r2=r2_score(y_test,y_pred_rbf)
mae = mean_absolute_error(y_test,y_pred_rbf)
rmse = np.sqrt(mean_squared_error(y_test,y_pred_rbf))

In [43]:
y_train_pred = svr_rbf.predict(X_train_scaled)
train_r2 = r2_score(y_train,y_train_pred)
print("Train R2:", round(train_r2,4))

Train R2: 0.9893


In [44]:
print("Test Metrics")
print("r2: ",r2)
print("mae: ",mae)
print("rmse: ",rmse)


Test Metrics
r2:  0.9810212072163411
mae:  0.026749953323994033
rmse:  0.07906050069782933


In [58]:
from sklearn.model_selection import cross_val_score
cv_scores = cross_val_score(svr_rbf,X_train_scaled,y_train,cv=10,scoring="r2",n_jobs=-1)
print("Fold Scores:")
print(cv_scores)
print()
print("Mean CV R2:",round(cv_scores.mean(),4))
print("Std CV R2:",round(cv_scores.std(),4))

Fold Scores:
[0.99020233 0.98320264 0.97824689 0.98302582 0.98598495 0.97348703
 0.97576778 0.98391988 0.97611726 0.94958267]

Mean CV R2: 0.978
Std CV R2: 0.0107


In [60]:
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import cross_val_score

cv = RepeatedKFold(
    n_splits=10,
    n_repeats=10,
    random_state=42
)

scores = cross_val_score(
    svr_rbf,
    X_train_scaled,
    y_train,
    cv=cv,
    scoring="r2",
    n_jobs=-1
)

print(scores.mean())
print(scores.std())

0.9786391822323816
0.008249855363211022


In [71]:
import optuna
def objective(trial):

    svr_model = SVR(

        kernel="rbf",

        C=trial.suggest_float(
            "C",
            1,
            1000,
            log=True
        ),

        gamma=trial.suggest_float(
            "gamma",
            0.0001,
            1,
            log=True
        ),

        epsilon=trial.suggest_float(
            "epsilon",
            0.001,
            0.5,
            log=True
        )
    )

    scores = cross_val_score(
        svr_model,
        X_train_scaled,
        y_train,
        cv=10,
        scoring="r2",
        n_jobs=-1
    )

    return scores.mean()

In [ ]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=50
)

[I 2026-06-11 16:40:33,976] A new study created in memory with name: no-name-be29661f-4dd5-4e70-82db-c78fd13e5fbe
[I 2026-06-11 16:40:44,660] Trial 0 finished with value: 0.7916109301657722 and parameters: {'C': 77.15882650025249, 'gamma': 0.0008187947831595646, 'epsilon': 0.004258447048145278}. Best is trial 0 with value: 0.7916109301657722.
[I 2026-06-11 16:41:18,249] Trial 1 finished with value: 0.9716785298792037 and parameters: {'C': 502.31143536936145, 'gamma': 0.025246667936974335, 'epsilon': 0.010200648450466823}. Best is trial 1 with value: 0.9716785298792037.
[I 2026-06-11 16:41:19,584] Trial 2 finished with value: 0.7892221090318012 and parameters: {'C': 3.860876881340512, 'gamma': 0.003384137743114767, 'epsilon': 0.008260332417507704}. Best is trial 1 with value: 0.9716785298792037.
[I 2026-06-11 16:41:20,508] Trial 3 finished with value: 0.46645395810211027 and parameters: {'C': 4.230757781005651, 'gamma': 0.00031105579258193316, 'epsilon': 0.04784119510595172}. Best is tr

: 

In [61]:
df=pd.read_csv("../data/DataWithDescriptors.csv")
df.columns

Index(['Unnamed: 0', 'IL NAME', 'CATION NAME', 'ANION NAME', 'Cation_SMILES',
       'Anion_SMILES', 'Temperature (0 C)', 'Pressure (bar)',
       'CO2 capacity (mol CO2/mol IL)', 'Cation_Family',
       ...
       'An_fr_sulfide', 'An_fr_sulfonamd', 'An_fr_sulfone',
       'An_fr_term_acetylene', 'An_fr_tetrazole', 'An_fr_thiazole',
       'An_fr_thiocyan', 'An_fr_thiophene', 'An_fr_unbrch_alkane',
       'An_fr_urea'],
      dtype='str', length=444)

In [62]:
features=pd.read_excel("../data/RF_Top20_Features.xlsx")

In [63]:
rf_features = features.iloc[:,0].tolist()

print(rf_features)
print(len(rf_features))

['Pressure (bar)', 'Temperature (0 C)', 'Cat_MolLogP', 'Cat_EState_VSA2', 'An_TPSA', 'Cat_VSA_EState3', 'An_AvgIpc', 'An_SlogP_VSA1', 'Cat_TPSA', 'Cat_EState_VSA9', 'Cat_BCUT2D_LOGPLOW', 'Cat_NumHDonors', 'Cat_SMR_VSA5', 'Cat_SMR_VSA1', 'Cat_PEOE_VSA1', 'Cat_NHOHCount', 'Cat_EState_VSA8', 'Cat_Chi4v', 'An_BalabanJ', 'An_PEOE_VSA3']
20


In [64]:
ranged_df=df[(df["Temperature (0 C)"] >= 25) &(df["Temperature (0 C)"] <= 30) &(df["Pressure (bar)"] >= 0.5) &(df["Pressure (bar)"] <= 1.5)].copy()

In [65]:
X_ranged = ranged_df[rf_features]

X_ranged_scaled = scaler.transform(X_ranged)

In [66]:
ranged_df["SVR_Pred"] = svr_rbf.predict(
    X_ranged_scaled
)

In [68]:
rf_ranking = (ranged_df.groupby("IL NAME")["SVR_Pred"].mean().reset_index())

In [70]:
top5_rf = rf_ranking.sort_values(by="SVR_Pred",ascending=False).head(10)

print(top5_rf)

          IL NAME  SVR_Pred
68    [TETAH][Pz]  2.040207
67    [TETAH][Py]  1.720325
66    [TETAH][Im]  1.711370
69    [TETAH][Tz]  1.510328
56  [P66614][Ala]  1.289982
57  [P66614][Gly]  1.249906
1     [AmemI][Br]  1.231372
71  [aP4443][Ala]  1.010093
58  [P66614][Ile]  0.960178
62  [P66614][Sar]  0.920146
